In [8]:
#Imports:
import os
import glob
import pickle
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
# 1. Définition du chemin vers vos fichiers .pkl
# Assurez-vous que ce chemin correspond à l'endroit où vous lancez le notebook
chemin_dossier_pkl = 'vectorisation-du-texte/output/'
fichiers_pkl = glob.glob(os.path.join(chemin_dossier_pkl, '*_FINAL.pkl'))

if not fichiers_pkl:
    print(f"Erreur : Aucun fichier .pkl trouvé dans {chemin_dossier_pkl}")
else:
    print(f"{len(fichiers_pkl)} fichiers de configuration trouvés.")

24 fichiers de configuration trouvés.


In [3]:
# 2. Variables pour mémoriser le meilleur modèle
meilleur_score_global = 0
meilleur_fichier_global = ""
meilleurs_parametres_global = {}
meilleur_modele_global = None
meilleur_X_test = None
meilleur_y_test = None

In [4]:
# 3. La grille des hyperparamètres exigée par le sujet (L1, L2, Elastic-Net)
# On la divise en deux dictionnaires pour éviter les erreurs/warnings avec l1_ratio
parametres_grille = [
    # Grille pour L1 et L2
    {'penalty': ['l1', 'l2'], 'C': [0.1, 1, 10], 'solver': ['saga']},
    # Grille pour Elastic-Net (qui a besoin du paramètre l1_ratio)
    {'penalty': ['elasticnet'], 'C': [0.1, 1, 10], 'solver': ['saga'], 'l1_ratio': [0.25, 0.5, 0.75]}
]

In [5]:
# 4. La grande boucle : on teste chaque fichier !
for chemin_fichier in fichiers_pkl:
    nom_fichier = os.path.basename(chemin_fichier)
    print(f"➡️ Test de la configuration : {nom_fichier}")
    
    # Chargement des données du fichier
    with open(chemin_fichier, 'rb') as f:
        data = pickle.load(f)
        
    X = data['X_normalized']
    y = data['target']
    
    # Séparation 80/20 avec le même random_state partout pour que ce soit comparable
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Création du modèle et de la recherche sur grille
    lr = LogisticRegression(max_iter=1000, random_state=42)
    grid = GridSearchCV(lr, parametres_grille, cv=5, n_jobs=-1, scoring='accuracy')
    
    # On ignore les warnings liés aux calculs complexes du solver 'saga'
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        grid.fit(X_train, y_train)
        
    score_actuel = grid.best_score_
    print(f"   Score : {score_actuel:.2%} | Meilleurs params locaux : {grid.best_params_}")
    
    # Si cette combinaison est meilleure que le record actuel, on la sauvegarde !
    if score_actuel > meilleur_score_global:
        meilleur_score_global = score_actuel
        meilleur_fichier_global = nom_fichier
        meilleurs_parametres_global = grid.best_params_
        meilleur_modele_global = grid.best_estimator_
        # On garde aussi les données de test pour afficher le rapport final
        meilleur_X_test = X_test 
        meilleur_y_test = y_test

➡️ Test de la configuration : config_L0_S0_LEM0_NG1_FINAL.pkl
   Score : 87.28% | Meilleurs params locaux : {'C': 1, 'penalty': 'l2', 'solver': 'saga'}
➡️ Test de la configuration : config_L0_S0_LEM0_NG2_FINAL.pkl
   Score : 88.77% | Meilleurs params locaux : {'C': 10, 'penalty': 'l2', 'solver': 'saga'}
➡️ Test de la configuration : config_L0_S0_LEM0_NG3_FINAL.pkl
   Score : 88.50% | Meilleurs params locaux : {'C': 10, 'l1_ratio': 0.25, 'penalty': 'elasticnet', 'solver': 'saga'}
➡️ Test de la configuration : config_L0_S0_LEM1_NG1_FINAL.pkl
   Score : 88.63% | Meilleurs params locaux : {'C': 1, 'penalty': 'l2', 'solver': 'saga'}
➡️ Test de la configuration : config_L0_S0_LEM1_NG2_FINAL.pkl
   Score : 89.65% | Meilleurs params locaux : {'C': 10, 'penalty': 'l2', 'solver': 'saga'}
➡️ Test de la configuration : config_L0_S0_LEM1_NG3_FINAL.pkl
   Score : 89.58% | Meilleurs params locaux : {'C': 10, 'penalty': 'l2', 'solver': 'saga'}
➡️ Test de la configuration : config_L0_S1_LEM0_NG1_FINAL.

In [7]:
# 5. Affichage du résultat final !
print("RÉSULTAT FINAL - LE MEILLEUR SYSTÈME EST :")
print(f"📍 Fichier de prétraitement gagnant : {meilleur_fichier_global}")
print(f"⚙️ Meilleurs hyperparamètres (Modèle) : {meilleurs_parametres_global}")
print(f"🎯 Précision (Score CV)             : {meilleur_score_global:.2%}")

# Rapport détaillé sur le jeu de test de cette meilleure configuration
y_pred_final = meilleur_modele_global.predict(meilleur_X_test)
print("\nRapport de classification du meilleur modèle sur les données de test :")
print(classification_report(meilleur_y_test, y_pred_final))

RÉSULTAT FINAL - LE MEILLEUR SYSTÈME EST :
📍 Fichier de prétraitement gagnant : config_L1_S0_LEM1_NG2_FINAL.pkl
⚙️ Meilleurs hyperparamètres (Modèle) : {'C': 10, 'penalty': 'l2', 'solver': 'saga'}
🎯 Précision (Score CV)             : 90.93%

Rapport de classification du meilleur modèle sur les données de test :
              precision    recall  f1-score   support

     négatif       0.86      0.93      0.89       188
     positif       0.92      0.84      0.88       182

    accuracy                           0.89       370
   macro avg       0.89      0.89      0.89       370
weighted avg       0.89      0.89      0.89       370



In [ ]:
#JUSTE POUR AVOIR CLASSEMENT GENERAL


# Liste pour stocker les résultats de TOUTES les configurations
tous_les_resultats = []

print("Création du classement en cours (cela peut prendre quelques minutes)...\n")

# La grande boucle d'évaluation pour le classement
for i, chemin_fichier in enumerate(fichiers_pkl, 1):
    nom_fichier = os.path.basename(chemin_fichier)
    
    # Chargement
    with open(chemin_fichier, 'rb') as f:
        data = pickle.load(f)
        
    X_encours = data['X_normalized']
    y_encours = data['target']
    
    X_train_encours, X_test_encours, y_train_encours, y_test_encours = train_test_split(X_encours, y_encours, test_size=0.2, random_state=42)
    
    # Entraînement
    lr_encours = LogisticRegression(max_iter=1000, random_state=42)
    grid_encours = GridSearchCV(lr_encours, parametres_grille, cv=5, n_jobs=-1, scoring='accuracy')
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        grid_encours.fit(X_train_encours, y_train_encours)
        
    # Sauvegarde du MEILLEUR résultat pour CE fichier précis
    tous_les_resultats.append({
        'Configuration': nom_fichier.replace('_FINAL.pkl', ''),
        'Score (%)': round(grid_encours.best_score_ * 100, 2),
        'Pénalité': grid_encours.best_params_['penalty'].upper(),
        'Meilleurs Params': str(grid_encours.best_params_)
    })

# Création du classement final (Leaderboard)
print("🏆 CLASSEMENT GÉNÉRAL DES 24 CONFIGURATIONS 🏆")

# Utilisation de Pandas pour faire un beau tableau trié
df_resultats = pd.DataFrame(tous_les_resultats)
df_resultats = df_resultats.sort_values(by='Score (%)', ascending=False).reset_index(drop=True)

# Ajustement de l'affichage pour tout voir dans la console
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# On décale l'index de 1 pour que le top 1 commence à 1 (et non 0)
df_resultats.index = df_resultats.index + 1 

# Affichage du tableau
display(df_resultats[['Configuration', 'Score (%)', 'Pénalité', 'Meilleurs Params']])

# Conclusion
meilleur = df_resultats.iloc[0]
print("\n" + "*"*80)
print(f"🎯 CONCLUSION : La configuration gagnante est '{meilleur['Configuration']}' ")
print(f"avec un score de {meilleur['Score (%)']}% en utilisant la régularisation {meilleur['Pénalité']}.")
print("*"*80)

Création du classement en cours (cela peut prendre quelques minutes)...

🏆 CLASSEMENT GÉNÉRAL DES 24 CONFIGURATIONS 🏆


,Configuration,Score (%),Pénalité,Meilleurs Params
1,config_L1_S0_LEM1_NG2,90.93,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
2,config_L1_S0_LEM0_NG3,90.87,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
3,config_L1_S0_LEM0_NG2,90.73,ELASTICNET,"{'C': 10, 'l1_ratio': 0.5, 'penalty': 'elasticnet', 'solver': 'saga'}"
4,config_L1_S0_LEM1_NG3,90.53,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
5,config_L1_S0_LEM1_NG1,89.85,L2,"{'C': 1, 'penalty': 'l2', 'solver': 'saga'}"
6,config_L0_S0_LEM1_NG2,89.65,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
7,config_L1_S1_LEM1_NG3,89.58,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
8,config_L0_S0_LEM1_NG3,89.58,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
9,config_L1_S0_LEM0_NG1,89.45,L2,"{'C': 10, 'penalty': 'l2', 'solver': 'saga'}"
10,config_L1_S1_LEM0_NG3,89.38,ELASTICNET,"{'C': 10, 'l1_ratio': 0.25, 'penalty': 'elasticnet', 'solver': 'saga'}"



********************************************************************************
🎯 CONCLUSION : La configuration gagnante est 'config_L1_S0_LEM1_NG2' 
avec un score de 90.93% en utilisant la régularisation L2.
********************************************************************************
